# DSO 576 Module 6 - Cleaning walkthrough

**Project:** cpg-global-entry-strategy  
**Student:** Antonio  

This notebook runs `src/clean_passport.py` from the original Passport files in `data/raw/`, shows the cleaning report, and runs the five record checks for checklist item 12.

**To reproduce:** Kernel → Restart, then Run All. Before committing, use **Clear All Outputs** (the real Euromonitor values must not go into the public repo).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import clean_passport as cp

results = cp.main()   # re-creates every file in output/, reports/, sample/

## Cleaning report (generated by the script)

In [ ]:
display(Markdown((ROOT / "reports" / "cleaning_report.md").read_text(encoding="utf-8")))

## Quick look at the clean tables

In [ ]:
for name in ["market_sizes", "retail_channels", "pack_type", "pack_size"]:
    df = results[name]
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
results["market_sizes"].head()

In [ ]:
# Main comparison metric: CAGR 2020-2025 for countries, Coffee retail volume
g = results["growth"]
(g[(g.geo_level == "Country") & (g.data_type == "Retail Volume")]
   .sort_values("cagr_2020_2025", ascending=False))

## Item 12: Five record checks

**Rule from the assignment:** write the expected result and the reason *before* looking at the output.

1. Open the original `.xls` file in Excel (do not save it) and find the row.
2. Type the expected clean value into `expected=` and your reason into `reason=`.
3. Then run the cell. `expected=None` means you have not filled it in yet.

Use `float("nan")` as the expected value when you expect the clean value to be missing, and `"DROPPED"` when you expect the row to be removed.

In [ ]:
import math

RAW_NAME = {"geography": "Geography", "category": "Category", "data_type": "Data Type",
            "outlet_type": "Outlet Type", "pack_type": "Pack Type", "pack_size": "Pack Size"}


def raw_value(dataset, filters, year):
    """Return the cell exactly as it appears in the original Excel file (as text)."""
    df, _ = cp.load_raw(dataset)
    mask = pd.Series(True, index=df.index)
    for col, val in filters.items():
        mask &= df[RAW_NAME[col]].str.strip() == val
    hits = df.loc[mask, str(year)]
    return repr(hits.iloc[0]) if len(hits) else "(not found)"


def clean_value(dataset, filters, year, column="value"):
    df = results[dataset]
    mask = df["year"] == year
    for col, val in filters.items():
        mask &= df[col] == val
    hits = df.loc[mask, column]
    return "DROPPED" if hits.empty else hits.iloc[0]


def check(n, dataset, filters, year, expected, reason, column="value", raw_dataset=None):
    if expected is None:
        print(f"Record {n}: fill in `expected` and `reason` first.")
        return None
    original = raw_value(raw_dataset or dataset, filters, year)
    actual = clean_value(dataset, filters, year, column)
    if isinstance(expected, str) or isinstance(actual, str):
        ok = expected == actual
    elif isinstance(expected, float) and math.isnan(expected):
        ok = pd.isna(actual)
    else:
        ok = (not pd.isna(actual)) and abs(actual - expected) < 0.01
    return {"record": n, "dataset": dataset, "filters": filters, "year": year,
            "original": original, "expected": expected, "actual": actual,
            "match": "YES" if ok else "NO", "reason": reason}


checks = [
    # 1. Unit rescale: JPY billion -> JPY million
    check(1, "market_sizes",
          {"geography": "Japan", "category": "Coffee", "data_type": "Retail Value RSP"}, 2025,
          expected=461100.0,
          reason="Raw 461.1 is in JPY billion; rule rescales all values to millions (x1000)"),
    # 2. '-' must become missing, not 0
    check(2, "market_sizes",
          {"geography": "India", "category": "RTD Coffee", "data_type": "Off-trade Volume"}, 2025,
          expected=float("nan"),
          reason="Raw cell is '-' (not available); rule converts '-' to missing, never 0"),
    # 3. Trailing space in label 'PET Jars '
    check(3, "pack_type",
          {"geography": "World", "category": "Coffee", "pack_type": "PET Jars"}, 2025,
          expected=110.8,
          reason="Label 'PET Jars ' had a trailing space; stripping it must not change the value"),
    # 4. Merge calculation: e-commerce share x market volume
    check(4, "channel_volume",
          {"geography": "World", "category": "RTD Coffee", "data_type": "Off-trade Volume",
           "outlet_type": "Retail E-Commerce"}, 2025,
          expected=311.69,
          reason="5.5% share x 5,667.1 million litres World RTD volume = 311.69",
          column="channel_volume", raw_dataset="retail_channels"),
    # 5. Redundant 'Total' channel row should be dropped
    check(5, "retail_channels",
          {"geography": "USA", "category": "Coffee", "data_type": "Retail Volume",
           "outlet_type": "Total"}, 2025,
          expected="DROPPED",
          reason="'Total' equals 'Retail Channels' (both 100); duplicate row removed"),
]
pd.DataFrame([c for c in checks if c])

## Item 13: Reproducibility

After **Restart + Run All**, the fingerprints below must match `reports/output_hashes.txt` from your submitted commit.

In [ ]:
print((ROOT / "reports" / "output_hashes.txt").read_text())